In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from fundus_data_toolkit.functional import open_image
from jppype import Mosaic, vscode_theme
import tqdm

from fundus_odmac_toolkit.models.segmentation import segment
from fundus_toolkits import FundusData
from fundus_toolkits.utils.data_io import most_common_image_ext
from fundus_vessels_toolkit import VTree
from fundus_vessels_toolkit.models import segment_av
from fundus_vessels_toolkit.pipelines.avseg_to_tree import GNNAVSegToTree, NaiveAVSegToTree, AVSegToTree
from fundus_vessels_toolkit.segment_to_graph.tree_topology import TreeTopology, optimal_lines
from fundus_vessels_toolkit.segment_to_graph.vbranch_digraph import VBranchDigraph
from fundus_vessels_toolkit.utils.jppype import draw_graph, draw_tree, draw_trees

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

## Load Image and Segment AV, OD, Macula


In [3]:
BL = Path("/home/gaby/Lab/DATA/Fundus/CLSA/BL/")
F1 = Path("/home/gaby/Lab/DATA/Fundus/CLSA/F1/")
RAW = "1-images"
AV_Auto = "2-av-pred_Automorph"
AV_FVT = "2-av-pred_FVT"
AV_VASCX = "2-av-pred_VascX"
OD = "2-od"
MACULA = "2-mac"

av2tree_naive = NaiveAVSegToTree()
av2tree_heur = AVSegToTree()
av2tree_gnn = GNNAVSegToTree()

IMGS = sorted({_.stem for _ in (BL / RAW).glob(f"*.jpeg")} & {_.stem for _ in (F1 / RAW).glob(f"*.jpeg")})
len(IMGS)

41153

In [4]:
from matplotlib.pylab import f

from fundus_vessels_toolkit.models.wrappers import vascx
from fundus_vessels_toolkit.models.wrappers.vascx import vascx_segment_av

for img in tqdm.tqdm(IMGS):
    for path in [BL, F1]:
        fvt_path = path / AV_FVT / (img + ".png")
        vascx_path = path / AV_VASCX / (img + ".png")
        if fvt_path.exists() and vascx_path.exists():
            continue
        fundus = FundusData(image=path / RAW / (img + ".jpeg"))
        if not fvt_path.exists():
            segment_av(fundus)
            fundus.write_image(av=fvt_path)
        if not vascx_path.exists():
            vascx_segment_av(fundus)
            fundus.write_image(av=vascx_path)


  0%|          | 0/41153 [00:00<?, ?it/s]/home/gaby/Lab/Libs/fundus-toolkits-common/src/fundus_toolkits/fundus_data.py:581: RuntimeWarning: The computed ROI mask is smaller than 60% of the image size and might be invalid.
  fundus_mask_ = fundus_ROI(fundus)  # Compute the fundus mask from the fundus image
/home/gaby/.conda/envs/lab/lib/python3.13/site-packages/monai/inferers/utils.py:226: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:309.)
  win_data = torch.cat([inputs[win_slice] for win_slice in unravel_slice]).to(sw_device)
/home/gaby/.conda/envs/lab/lib/python3.13/site-packages/monai/inferers/utils.py:370: UserWarning: Using a non-tuple sequence for multid

ValueError: zero-size array to reduction operation minimum which has no identity

In [ ]:
STOP

In [ ]:
IMG = IMGS[50]
seg_model = "fvt"


def load_fundus(img, path, seg_model="fvt"):
    img_png = img + ".png"
    av = {"fvt": AV_FVT, "automorph": AV_Auto}[seg_model]
    fundus = FundusData(image=path / RAW / (img + ".jpeg"), od=path / OD / img_png, macula=path / MACULA / img_png)
    if (path / av / img_png).exists():
        fundus.update(av=path / av / img_png, inplace=True)
    else:
        segment_av(fundus)
        fundus.write_image(av=path / av / img_png, on_exists="skip")
    return fundus.remove_od_from_vessels()


fundus_bl = load_fundus(IMG, BL, seg_model=seg_model)
fundus_f1 = load_fundus(IMG, F1, seg_model=seg_model)
m = Mosaic((2, 3), cols_titles=["input", "GNN", "Heuristique"], rows_titles=["BL", "F1"], cell_height=500)

fundus_bl.draw(view=m[0, 0])
draw_trees(av2tree_naive(fundus_bl), view=m[0, 0], bspline_dir=True, interactive=True)
m[0, 1].add_image(fundus_bl.image)
draw_trees(av2tree_gnn(fundus_bl), view=m[0, 1], bspline_dir=True, interactive=True)
m[0, 2].add_image(fundus_bl.image)
draw_trees(av2tree_heur(fundus_bl), view=m[0, 2], bspline_dir=True, interactive=True)


fundus_f1.draw(view=m[1, 0])
draw_trees(av2tree_naive(fundus_f1), view=m[1, 0], bspline_dir=True, interactive=True)
m[1, 1].add_image(fundus_f1.image)
draw_trees(av2tree_gnn(fundus_f1), view=m[1, 1], bspline_dir=True, interactive=True)
m[1, 2].add_image(fundus_f1.image)
draw_trees(av2tree_heur(fundus_f1), view=m[1, 2], bspline_dir=True, interactive=True)

m

[ WARN:0@4.467] global loadsave.cpp:1671 imencodeWithMetadata Unsupported depth image for selected encoder is fallbacked to CV_8U.
/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/vascular_data_objects/vtree.py:414: UserWarning: The geometric data contains duplicated nodes coordinates.
  super().__init__(
/home/gaby/Lab/Src/fundus-vessels-toolkit/src/fundus_vessels_toolkit/vascular_data_objects/vtree.py:414: UserWarning: The geometric data contains duplicated nodes coordinates.
  super().__init__(


GridBox(children=(HTML(value='<span/>'), HTML(value='<h3 style="text-align: center;">input</h3>'), HTML(value=…